[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch07-frameworks/02-do-thi-va-bien-dich.ipynb)

# Đồ thị, hợp nhất và khi nào biên dịch hoàn vốn

Một framework chỉ hợp nhất được những phép mà nó thấy cùng lúc. Thực thi tức thời đưa từng phép cho
bộ điều phối rồi đi tiếp. Bắt đồ thị (graph capture) giữ lại cả chuỗi, để trình biên dịch tối ưu
trước khi chạy. Sổ tay này đọc chính đồ thị mà PyTorch bắt được, đếm cơ hội hợp nhất trong đó, đo một
lần hợp nhất thật bằng `torch.compile`, rồi dùng công thức của chương để hỏi khi nào khoản biên dịch
được bù lại.

**Bạn sẽ làm:**
1. bắt đồ thị tính toán (computational graph) của một khối nhỏ bằng `torch.fx`, đọc từng đỉnh cùng
   hình dạng tensor của nó, rồi loại mã chết;
2. đếm các chuỗi phép theo phần tử hợp nhất được, cùng số lần nạp kernel và số byte trước và sau;
3. xem lần theo vết (tracing) thất bại im lặng trước một nhánh phụ thuộc dữ liệu;
4. đo một chuỗi tám phép theo phần tử, chạy tức thời và chạy sau `torch.compile`;
5. tính lợi ích biên dịch và điểm hoà vốn với bảng giả định của chương.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import operator
import warnings

import pandas as pd
import torch.fx as fx
from torch import nn
from torch.fx.passes.shape_prop import ShapeProp

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # bốn màu cố định cho các chuỗi số liệu


def thoi_gian(viec, lan=5):
    """Thời gian nhỏ nhất của `lan` lần chạy (giây): ít bị nhiễu bởi việc khác trên máy nhất."""
    viec()
    tot = float("inf")
    for _ in range(lan):
        t0 = time.perf_counter()
        viec()
        tot = min(tot, time.perf_counter() - t0)
    return tot


print("sẵn sàng")

## 1. Đồ thị mà framework nhìn thấy

Khối dưới đây là một mảnh MLP: một tầng dày đặc, một hàm kích hoạt GELU xấp xỉ bằng
$h \cdot \sigma(1.702\,h)$, một tầng dày đặc nữa, rồi cộng phần dư và qua ReLU. Ở giữa có một dòng
tính một con số mà không ai dùng tới, để xem trình biên dịch làm gì với nó.

`torch.fx.symbolic_trace` chạy `forward` với tensor **ký hiệu** thay cho tensor thật: mỗi phép không
tính gì cả mà chỉ thêm một đỉnh vào đồ thị, đúng như pha dựng của đồ thị tĩnh (static graph) mà
chương mô tả. Sau đó `ShapeProp` cho một tensor mẫu chạy qua đồ thị để ghi hình dạng ở từng đỉnh.

In [ ]:
class Khoi(nn.Module):
    """Linear, GELU xấp xỉ, Linear, cộng phần dư rồi ReLU."""

    def __init__(self, d=64, rong=256):
        super().__init__()
        self.fc1 = nn.Linear(d, rong)
        self.fc2 = nn.Linear(rong, d)

    def forward(self, x):
        h = self.fc1(x)
        h = h * torch.sigmoid(1.702 * h)   # GELU xấp xỉ: ba phép theo phần tử
        thong_ke = h.mean() * 0             # tính ra rồi bỏ đó: không ai dùng
        h = self.fc2(h)
        return torch.relu(h + x)            # phần dư rồi ReLU: hai phép theo phần tử


dat_hat_giong()
khoi = Khoi()
gm = fx.symbolic_trace(khoi)
x_mau = torch.randn(32, 64)
ShapeProp(gm).propagate(x_mau)

THEO_PHAN_TU = {operator.mul, operator.add, operator.sub, torch.sigmoid, torch.relu, torch.tanh}


def loai(dinh):
    if dinh.op == "call_module" and isinstance(gm.get_submodule(dinh.target), nn.Linear):
        return "GEMM"
    if dinh.op == "call_function" and dinh.target in THEO_PHAN_TU:
        return "theo phần tử"
    if dinh.op in ("call_method", "call_function"):
        return "rút gọn" if dinh.target in ("mean", "sum") else "khác"
    return dinh.op   # placeholder (đầu vào) hoặc output


hang = []
for dinh in gm.graph.nodes:
    tm = dinh.meta.get("tensor_meta")
    hinh = tuple(tm.shape) if tm is not None and hasattr(tm, "shape") else ""
    hang.append((dinh.name, loai(dinh), str(getattr(dinh.target, "__name__", dinh.target)),
                 ", ".join(str(a) for a in dinh.args), str(hinh), len(dinh.users)))
print(pd.DataFrame(hang, columns=["đỉnh", "loại", "phép", "đầu vào", "hình dạng", "số nơi dùng"]).to_string(index=False))

Đó là một đồ thị có hướng không chu trình: đỉnh là phép toán, cung là phụ thuộc dữ liệu. Cột "số nơi
dùng" cho biết kết quả của mỗi đỉnh được bao nhiêu đỉnh khác đọc. Đỉnh `mul_2` không có nơi dùng nào,
nên có thể bỏ mà kết quả không đổi. Đó là loại mã chết (dead code elimination), một trong những tối
ưu trước khi chạy mà chương kể ra. Ở chế độ tức thời, dòng ấy vẫn chạy mỗi lần, vì framework không
biết trước rằng sẽ không ai dùng nó.

In [ ]:
so_dinh_truoc = len(gm.graph.nodes)
gm.graph.eliminate_dead_code()
gm.recompile()
so_dinh_sau = len(gm.graph.nodes)
print(f"số đỉnh: {so_dinh_truoc} → {so_dinh_sau}")
assert so_dinh_truoc - so_dinh_sau == 2                     # `mean` và phép nhân với 0
assert torch.allclose(gm(x_mau), khoi(x_mau))               # bỏ đi rồi, kết quả vẫn y hệt
print("mã Python mà đồ thị sinh lại sau khi loại mã chết:")
print(gm.code)

## 2. Đếm cơ hội hợp nhất trên đồ thị

Hai phép theo phần tử liền nhau hợp nhất được thành một kernel nếu kết quả giữa của chúng không được
ai khác đọc: giá trị đó khi ấy chỉ cần nằm trong thanh ghi, không cần ghi ra bộ nhớ rồi đọc lại. Ô
dưới gom các đỉnh theo phần tử thành nhóm theo đúng quy tắc đó, rồi đếm hai thứ mà hợp nhất giảm
được: số lần nạp kernel, và số byte đọc ghi. Cách đếm byte là mô hình lý tưởng của chương: mỗi phép
đọc mỗi tensor đầu vào một lần và ghi đầu ra một lần, mỗi giá trị FP32 4 byte.

**Dự đoán:** khối này có 7 phép tính toán. Hợp nhất xong còn bao nhiêu lần nạp kernel?

In [ ]:
BYTE = sach(4, nguon="4 byte → 1 byte")   # FP32
dinh = [d for d in gm.graph.nodes]
cha = {d: d for d in dinh}


def goc(d):
    while cha[d] is not d:
        d = cha[d]
    return d


for d in dinh:
    if loai(d) != "theo phần tử":
        continue
    for a in d.all_input_nodes:
        # gộp với đỉnh sinh ra đầu vào nếu đỉnh ấy cũng theo phần tử và chỉ phép theo phần tử đọc nó
        if loai(a) == "theo phần tử" and all(loai(u) == "theo phần tử" for u in a.users):
            cha[goc(d)] = goc(a)

nhom = {}
for d in dinh:
    if loai(d) == "theo phần tử":
        nhom.setdefault(goc(d), []).append(d)


def so_byte(d):
    return int(np.prod(d.meta["tensor_meta"].shape)) * BYTE


ket_qua = []
for thanh_vien in nhom.values():
    tap = set(thanh_vien)
    roi = sum(sum(so_byte(a) for a in d.all_input_nodes) + so_byte(d) for d in thanh_vien)
    vao_ngoai = {a for d in thanh_vien for a in d.all_input_nodes if a not in tap}
    ra_ngoai = [d for d in thanh_vien if any(u not in tap for u in d.users)]
    gop = sum(so_byte(a) for a in vao_ngoai) + sum(so_byte(d) for d in ra_ngoai)
    ket_qua.append((" → ".join(d.name for d in thanh_vien), len(thanh_vien), roi, gop, roi / gop))
print(pd.DataFrame(ket_qua, columns=["nhóm hợp nhất", "số phép", "byte chạy rời", "byte hợp nhất", "giảm"])
      .to_string(index=False, formatters={"byte chạy rời": "{:,}".format, "byte hợp nhất": "{:,}".format,
                                          "giảm": "{:.2f}×".format}))

so_gemm = sum(loai(d) == "GEMM" for d in dinh)
nap_truoc = so_gemm + sum(len(v) for v in nhom.values())
nap_sau = so_gemm + len(nhom)
print(f"số lần nạp kernel: {nap_truoc} → {nap_sau}")
assert (nap_truoc, nap_sau) == (7, 4)

Mỗi nhóm đọc các đầu vào từ bên ngoài một lần và ghi kết quả cuối một lần. Nhóm thứ nhất đọc đầu ra
của `fc1` hai lần khi chạy rời (một lần cho phép nhân với 1.702, một lần cho phép nhân cuối), nhưng
chỉ một lần khi hợp nhất. Hai phép nhân ma trận vẫn là hai kernel riêng do thư viện BLAS lo, vì quy tắc
gom ở đây chỉ nối các phép theo phần tử với nhau.

### Mô hình của chương: k phép trên N phần tử

Với một chuỗi $k$ phép theo phần tử, mỗi phép một đầu vào, trên $N$ phần tử mỗi phần tử $b$ byte,
chạy rời tốn $2kNb$ byte và $k$ lần nạp, hợp nhất còn $2Nb$ byte và một lần nạp. Tỉ số của cả hai
là $k$, không phụ thuộc $N$ hay $b$. Napkin Math 1.1 áp mô hình này cho $y = \mathrm{relu}(x + \text{bias})$,
với khoản 15 µs cho mỗi lần nạp; chương còn tính một chuỗi 5 phép trên một tensor 100 MB.

In [ ]:
T_DP = sach(15, nguon="15 µs")   # µs mỗi lần nạp, mô hình minh hoạ của chương


def chuoi(k, N, b, hop_nhat):
    """(phụ trội µs, byte đọc ghi) của k phép theo phần tử, chạy rời hoặc hợp nhất."""
    lan_nap = 1 if hop_nhat else k
    return lan_nap * T_DP, 2 * lan_nap * N * b if not hop_nhat else 2 * N * b


N, b = 10**6, BYTE   # giá trị bất kỳ: mọi tỉ số dưới đây không phụ thuộc chúng
pt_roi, byte_roi = chuoi(2, N, b, hop_nhat=False)
pt_gop, byte_gop = chuoi(2, N, b, hop_nhat=True)
theo_sach("relu(x + bias) chạy tức thời: phụ trội (µs)", pt_roi, sach=30, trich="Total overhead: 30 μs.")
theo_sach("relu(x + bias) hợp nhất: phụ trội (µs)", pt_gop, sach=15, trich="Total overhead: 15 μs (2× speedup).")
theo_sach("tỉ số phụ trội", pt_roi / pt_gop, sach=2, trich="15 μs (2× speedup)")
print(f"lưu lượng chạy tức thời: {byte_roi / (N * b):.0f}·Nb byte, hợp nhất: {byte_gop / (N * b):.0f}·Nb byte")
theo_sach("tỉ số lưu lượng", byte_roi / byte_gop, sach=2, trich="(2× bandwidth efficiency)")

MB = 10**6
k5 = sach(5, nguon="5 phép · 100 MB")
tensor_100MB = sach(100, nguon="5 phép · 100 MB") * MB
_, roi5 = chuoi(k5, tensor_100MB, 1, hop_nhat=False)
_, gop5 = chuoi(k5, tensor_100MB, 1, hop_nhat=True)
theo_sach("5 phép chạy rời (GB)", roi5 / 10**9, sach=1, nguon="1 GB → 200 MB · 5×")
theo_sach("5 phép hợp nhất (MB)", gop5 / MB, sach=200, nguon="1 GB → 200 MB · 5×")
theo_sach("mức tiết kiệm băng thông", roi5 / gop5, sach=5, nguon="1 GB → 200 MB · 5×")

## 3. Lần theo vết và thất bại im lặng

Bắt đồ thị từ mã Python cần biết trước những phép nào sẽ chạy. Hàm dưới rẽ nhánh theo **giá trị** của
tensor: tổng dương thì nhân đôi, ngược lại thì đổi dấu.

* `torch.fx` dựng đồ thị bằng tensor ký hiệu, chưa có giá trị, nên nó không trả lời được câu hỏi
  `x.sum() > 0`. Nó từ chối, và báo lỗi.
* Lần theo vết thì chạy hàm với một đầu vào mẫu thật và ghi lại những phép đã thật sự chạy. Chương
  minh hoạ bằng `torch.jit.trace` của TorchScript, một công cụ nay đã ngừng phát triển cho dự án mới;
  ta dùng nó ở đây đúng với vai trò đó, để thấy rủi ro chương mô tả.

**Dự đoán:** vết bắt được với một đầu vào dương sẽ trả gì cho đầu vào $[-1, -1, -1]$? Đúng là
$[1, 1, 1]$.

In [ ]:
def re_nhanh(x):
    if x.sum() > 0:
        return x * 2
    return -x


try:
    fx.symbolic_trace(re_nhanh)
except Exception as loi:
    print(f"torch.fx từ chối: {type(loi).__name__}: {str(loi).splitlines()[0]}")

with warnings.catch_warnings(record=True) as canh_bao:
    warnings.simplefilter("always")
    vet = torch.jit.trace(re_nhanh, torch.ones(3))
for w in canh_bao:
    if w.category.__name__ == "TracerWarning":
        print(f"cảnh báo khi lần vết: {str(w.message).splitlines()[0][:110]}…")
        break
print("đồ thị mà vết giữ lại:")
print(vet.graph)

am = -torch.ones(3)
print("chạy tức thời:", re_nhanh(am).tolist())
print("chạy theo vết:", vet(am).tolist())
assert not torch.equal(vet(am), re_nhanh(am))   # vết vẫn chạy, chỉ là sai

Vết chỉ chứa một phép nhân với 2: nhánh còn lại không hề tồn tại trong đó. Nó vẫn chạy được với đầu
vào âm, vẫn cho ra những con số trông hợp lý, và chúng sai. Vì vậy chương nói rằng khi có luồng điều
khiển Python phụ thuộc dữ liệu, cảnh báo khi lần vết phải được coi là lỗi về tính đúng, chứ không
phải lời nhắc. Các trình biên dịch bắt đồ thị hiện đại xử lý chỗ này bằng ngắt đồ thị (graph break):
phần mã không bắt được thì quay về chạy tức thời, và đúng, nhưng vùng biên dịch ngắn lại.

## 4. Đo một lần hợp nhất thật

Hàm dưới là một chuỗi $k = 8$ phép theo phần tử, mỗi phép một đầu vào tensor, đúng mô hình $2kNb$ ở
phần 2. Ta đo nó ở bốn kích thước $N$, chạy tức thời và chạy sau `torch.compile`, trình biên dịch bắt
đồ thị của PyTorch 2. Đây là hàm duy nhất trong sổ tay được biên dịch. Lần gọi đầu tiên với một hình
dạng chưa gặp trả cả chi phí bắt, hạ và sinh mã, nên ta đo nó riêng.

**Dự đoán:** với $N$ lớn, bản biên dịch nhanh hơn bao nhiêu lần? Có tới 8 lần không?

In [ ]:
def tam_phep(x):
    """Tám phép theo phần tử nối tiếp nhau, mỗi phép đọc một tensor và ghi một tensor."""
    y = x * 0.5
    y = y + 0.25
    y = torch.relu(y)
    y = y - 0.1
    y = torch.abs(y)
    y = y * 3.0
    y = torch.clamp(y, max=4.0)
    y = y + 1.0
    return y


K_PHEP = 8
cac_N = [2**22, 2**18, 2**14, 2**10]     # lớn trước: lần biên dịch đầu là ở kích thước lớn nhất
dat_hat_giong()
dau_vao = {n_: torch.randn(n_) for n_ in cac_N}

tam_phep_bd = torch.compile(tam_phep)
lan_dau, t_tuc_thoi, t_bien_dich = {}, {}, {}
for n_ in cac_N:
    xn = dau_vao[n_]
    t0 = time.perf_counter()
    y_bd = tam_phep_bd(xn)                 # có thể kích hoạt bắt và biên dịch
    lan_dau[n_] = time.perf_counter() - t0
    assert torch.allclose(y_bd, tam_phep(xn))
    t_tuc_thoi[n_] = thoi_gian(lambda: tam_phep(xn))
    t_bien_dich[n_] = thoi_gian(lambda: tam_phep_bd(xn))

t0 = time.perf_counter()
tam_phep_bd(dau_vao[2**18].double())       # đổi kiểu dữ liệu: một chốt bảo vệ khác
lan_dau_fp64 = time.perf_counter() - t0

for n_ in cac_N:
    do_tren_may(f"N = 2^{int(np.log2(n_)):<2}  lần gọi đầu của bản biên dịch", lan_dau[n_] * 1e3, "ms")
do_tren_may("N = 2^18 FP64, lần gọi đầu", lan_dau_fp64 * 1e3, "ms")
for n_ in sorted(cac_N):
    do_tren_may(f"N = 2^{int(np.log2(n_)):<2}  tức thời / biên dịch, mỗi lần gọi",
                t_tuc_thoi[n_] / t_bien_dich[n_], "lần")

Hãy đọc các con số ⏱ ở trên theo hai cột.

* **Lần gọi đầu.** Lần đầu tiên là cả một lần biên dịch. Khi một hình dạng mới xuất hiện, chốt bảo vệ
  (guard) về hình dạng của mã đã biên dịch có thể hỏng, và môi trường chạy (runtime) phải biên dịch
  lại. PyTorch mặc định đánh dấu chiều đó là chiều động sau lần hỏng đầu tiên, nên các kích thước sau
  thường không phải biên dịch nữa. Đổi kiểu dữ liệu thì làm hỏng một chốt khác. Lần gọi nào nhanh
  ngang một lần gọi thường là lần gọi đã thoả mọi chốt.
* **Mỗi lần gọi sau đó.** Ô dưới vẽ tỉ số này theo $N$, cạnh tỉ số lý tưởng $k = 8$ của mô hình.

In [ ]:
Ns = sorted(cac_N)
tang_toc = [t_tuc_thoi[n_] / t_bien_dich[n_] for n_ in Ns]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogx(Ns, tang_toc, "o-", color=MAU[0], ms=8, base=2, label="đo được: tức thời / biên dịch")
ax.axhline(K_PHEP, color=MAU[1], ls="--", lw=1.2, label="lý tưởng: k = 8 (cả số lần nạp lẫn byte)")
ax.axhline(1, color="0.5", lw=1)
for n_, r in zip(Ns, tang_toc):
    ax.annotate(f"{r:.1f}×", (n_, r), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
ax.set_xticks(Ns, [f"$2^{{{int(np.log2(n_))}}}$" for n_ in Ns])
ax.set_ylim(0, K_PHEP * 1.25)
ax.set_xlabel("số phần tử N")
ax.set_ylabel("tăng tốc mỗi lần gọi")
ax.set_title("Hợp nhất tám phép theo phần tử (đo trên máy này)")
ax.legend(loc="center left", fontsize=8)
plt.tight_layout()
plt.show()

Với một chuỗi $k$ phép mỗi phép một đầu vào, hợp nhất chia cả số lần nạp lẫn lưu lượng lý tưởng cho
đúng $k$. Ở $N$ nhỏ, phần thắng thua là phụ trội: tám lần điều phối, so với một lần gọi mã đã biên
dịch vốn cũng có phụ trội riêng là việc kiểm tra các chốt bảo vệ. Ở $N$ lớn, phần thắng thua là lưu
lượng bộ nhớ. Tỉ số đo được thường nằm dưới $k$, vì hai lý do: kernel hợp nhất vẫn làm đủ tám phép tính
trên mỗi phần tử, và bộ nhớ đệm của CPU giữ được một phần các kết quả giữa mà mô hình lý tưởng tính là đi
ra bộ nhớ chính.

Lợi ích biên dịch của chương (phương trình 2) cần đúng ba con số vừa đo: chi phí biên dịch, thời gian
một lần chạy tức thời, và thời gian một lần chạy sau biên dịch. Điểm hoà vốn (phương trình 3) là

$$N_{\text{hoà vốn}} = \frac{T_{compile}}{T_{eager} - T_{compiled}}.$$

In [ ]:
N_lon = cac_N[0]
T_compile_do = lan_dau[N_lon] - t_bien_dich[N_lon]   # lần gọi đầu trừ đi một lần chạy thường
tiet_kiem = t_tuc_thoi[N_lon] - t_bien_dich[N_lon]
do_tren_may("chi phí biên dịch đo được, T_compile", T_compile_do, "s")
do_tren_may("tiết kiệm mỗi lần gọi ở N = 2^22", tiet_kiem * 1e3, "ms")
if tiet_kiem > 0:
    do_tren_may("số lần gọi để hoà vốn trên máy này", T_compile_do / tiet_kiem, "lần gọi")
else:
    print("Trên máy này bản biên dịch không nhanh hơn ở N = 2^22: biên dịch không bao giờ hoàn vốn.")

## 5. Khi nào biên dịch hoàn vốn: bảng giả định của chương

Bảng 4 của chương cho thông lượng giả định trên một GPU A100 với batch 32, ba chế độ chạy và thời gian
biên dịch của `torch.compile`. Chương nói rõ các số này là giả định để minh hoạ, không phải kết quả đo
benchmark. BERT, ViT và GPT-2 tính bằng mẫu mỗi giây.

In [ ]:
bang4 = {  # mô hình: (tức thời, torch.compile, TensorRT) mẫu/giây, và khoảng thời gian biên dịch (s)
    "ResNet-50": ((sach(1450, nguon="1.450 · 2.150 · 3.800 img/s"), sach(2150, nguon="1.450 · 2.150 · 3.800 img/s"),
                   sach(3800, nguon="1.450 · 2.150 · 3.800 img/s")),
                  (sach(15, nguon="3.800 img/s · 15–30 s"), sach(30, nguon="3.800 img/s · 15–30 s"))),
    "BERT-Base": ((sach(380, nguon="380 · 520 · 890 · 30–60 s"), sach(520, nguon="380 · 520 · 890 · 30–60 s"),
                   sach(890, nguon="380 · 520 · 890 · 30–60 s")),
                  (sach(30, nguon="380 · 520 · 890 · 30–60 s"), sach(60, nguon="380 · 520 · 890 · 30–60 s"))),
    "ViT-B/16": ((sach(620, nguon="620 · 950 · 1.650 · 25–45 s"), sach(950, nguon="620 · 950 · 1.650 · 25–45 s"),
                  sach(1650, nguon="620 · 950 · 1.650 · 25–45 s")),
                 (sach(25, nguon="620 · 950 · 1.650 · 25–45 s"), sach(45, nguon="620 · 950 · 1.650 · 25–45 s"))),
    "GPT-2 124M": ((sach(180, nguon="180 · 260 · 420 · 45–90 s"), sach(260, nguon="180 · 260 · 420 · 45–90 s"),
                    sach(420, nguon="180 · 260 · 420 · 45–90 s")),
                   (sach(45, nguon="180 · 260 · 420 · 45–90 s"), sach(90, nguon="180 · 260 · 420 · 45–90 s"))),
}
tt_compile = {m: v[0][1] / v[0][0] for m, v in bang4.items()}
tt_trt = {m: v[0][2] / v[0][0] for m, v in bang4.items()}
for m in bang4:
    print(f"{m:<11} torch.compile {tt_compile[m]:.2f}×   TensorRT {tt_trt[m]:.2f}×")
print(f"torch.compile: {min(tt_compile.values()):.2f}× đến {max(tt_compile.values()):.2f}×, "
      f"TensorRT: {min(tt_trt.values()):.2f}× đến {max(tt_trt.values()):.2f}×")
assert all(tt_trt[m] > tt_compile[m] > 1 for m in bang4)   # cùng một thứ tự cho cả bốn mô hình
theo_sach("ResNet-50: thông lượng cao hơn khi biên dịch (%)", 100 * (tt_compile["ResNet-50"] - 1), sach=48.3,
          nguon="48,3 phần trăm")

### Lợi ích biên dịch và điểm hoà vốn

Phương trình 2 của chương đặt tiết kiệm của mọi lần chạy sản xuất lên tử số, và mọi lần biên dịch xuống
mẫu số:

$$\text{lợi ích biên dịch} = \frac{N_{prod}\,(T_{eager} - T_{compiled})}{T_{compile} + N_{dev}\,T_{compile}}$$

Biên dịch khi lợi ích lớn hơn 1. Công thức giả định mã đã biên dịch còn hiệu lực giữa các lần sửa, và
mỗi lần biên dịch lại tốn như lần đầu. Với ResNet-50, chương dùng chi phí biên dịch 30 giây, đầu trên
của khoảng 15–30 s.

**Dự đoán:** cần bao nhiêu ảnh để 30 giây biên dịch được bù lại? Một nghìn, một trăm nghìn, hay một
triệu?

In [ ]:
def loi_ich(n_prod, n_dev, t_eager, t_compiled, t_compile):
    """Phương trình 2 của chương."""
    return n_prod * (t_eager - t_compiled) / (t_compile + n_dev * t_compile)


def hoa_von(t_eager, t_compiled, t_compile):
    """Phương trình 3 của chương."""
    return t_compile / (t_eager - t_compiled)


(r_eager, r_comp, r_trt), (_, T_COMPILE) = bang4["ResNet-50"]
n_hv = hoa_von(1 / r_eager, 1 / r_comp, T_COMPILE)
theo_sach("điểm hoà vốn của ResNet-50 (ảnh)", n_hv, sach=134_000, nguon="134.000 ảnh", tol=500)
print(f"kiểm lại: lợi ích ở đúng điểm hoà vốn, N_dev = 0: {loi_ich(n_hv, 0, 1 / r_eager, 1 / r_comp, T_COMPILE):.3f}")

lan_moi_gio = sach(10, nguon="10 lần mỗi giờ · 5 phút")
theo_sach("biên dịch lại mỗi giờ khi kiến trúc đổi liên tục (phút)", lan_moi_gio * T_COMPILE / 60, sach=5,
          nguon="10 lần mỗi giờ · 5 phút")

hang = []
for m, ((e, c_, _), (t_lo, t_hi)) in bang4.items():
    hang.append((m, hoa_von(1 / e, 1 / c_, t_lo), hoa_von(1 / e, 1 / c_, t_hi)))
print(pd.DataFrame(hang, columns=["mô hình", "hoà vốn, biên dịch nhanh", "hoà vốn, biên dịch chậm"])
      .to_string(index=False, formatters={"hoà vốn, biên dịch nhanh": "{:,.0f} mẫu".format,
                                          "hoà vốn, biên dịch chậm": "{:,.0f} mẫu".format}))

Bảng cuối là tính toán của sổ tay này trên số giả định của chương: mỗi mô hình hoà vốn sau vài chục tới
vài trăm nghìn mẫu, nằm gọn trong một lần huấn luyện dài.

### Ngộ nhận: phụ trội biên dịch không đáng kể

Chương đặt cùng các số đó vào một tình huống dựng mẫu: một thí nghiệm 10,000 ảnh với 10 lần sửa mã, mỗi
lần sửa làm đồ thị thay đổi và phải biên dịch lại 30 giây.

In [ ]:
n_anh = sach(10_000, trich="a 10,000-image experiment with 10 code changes")
n_sua = sach(10, trich="a 10,000-image experiment with 10 code changes")
t_eager_tn = n_anh / r_eager
t_bd_tn = n_anh / r_comp + n_sua * T_COMPILE
theo_sach("chế độ tức thời (s)", t_eager_tn, sach=6.9, nguon="6,9 s · 304,7 s · 44,2×")
theo_sach("chế độ biên dịch, tính cả biên dịch lại (s)", t_bd_tn, sach=304.7, nguon="6,9 s · 304,7 s · 44,2×")
theo_sach("biên dịch chậm hơn tức thời (lần)", t_bd_tn / t_eager_tn, sach=44.2, nguon="6,9 s · 304,7 s · 44,2×")
print(f"nếu đếm cả lần biên dịch đầu như mẫu số của phương trình 2: {n_anh / r_comp + (1 + n_sua) * T_COMPILE:.1f} s")
print(f"lợi ích biên dịch của thí nghiệm này: {loi_ich(n_anh, n_sua, 1 / r_eager, 1 / r_comp, T_COMPILE):.4f}")

Sách tính 10 lần biên dịch, một lần cho mỗi lần sửa. Mẫu số của phương trình 2 đếm thêm lần biên dịch đầu
tiên, nên theo đúng công thức là 11 lần và khoảng 334.7 s. Hai cách đếm cho hai con số khác nhau, nhưng
cùng một kết luận: lợi ích biên dịch nhỏ hơn 1 hàng trăm lần, và nhóm nên ở lại chế độ tức thời. Đó là
công thức lợi ích biên dịch đọc theo chiều ngược: khi số lần chạy phát triển lớn, mẫu số phình ra.

### Ba vùng: nghiên cứu, huấn luyện, phục vụ

Vẽ tổng thời gian theo số mẫu đã chạy: độ dốc của mỗi đường là chi phí một mẫu, độ cao ban đầu là phụ
trội biên dịch. Bảng 4 không ghi thời gian biên dịch của TensorRT, chỉ nói nó lâu hơn. Ở đây ta **giả
định** 10 phút, chỉ để vẽ điểm giao thứ hai; TensorRT còn chỉ dùng cho suy luận.

In [ ]:
T_TRT_GIA_DINH = 600.0   # giả định của sổ tay này, không phải số của sách
n = np.logspace(2, 8, 400)
duong = {
    "tức thời": n / r_eager,
    f"torch.compile, biên dịch {T_COMPILE:g} s": T_COMPILE + n / r_comp,
    "TensorRT, biên dịch 600 s (giả định)": T_TRT_GIA_DINH + n / r_trt,
}
fig, ax = plt.subplots(figsize=(7, 4.2))
for (ten, y), mau, ls in zip(duong.items(), MAU, ("-", "--", ":")):
    ax.loglog(n, y, color=mau, lw=2, ls=ls, label=ten)
giao2 = hoa_von(1 / r_comp, 1 / r_trt, T_TRT_GIA_DINH - T_COMPILE)
for nn, nhan, lech in ((n_hv, f"hoà vốn ≈ {round(n_hv, -3):,.0f} ảnh", (-150, 22)),
                       (giao2, f"giao thứ hai ≈ {giao2 / 1e6:.1f} triệu ảnh (giả định)", (10, -22))):
    ax.plot(nn, T_COMPILE + nn / r_comp, "o", color="0.2", ms=6)
    ax.annotate(nhan, (nn, T_COMPILE + nn / r_comp), xytext=lech, textcoords="offset points", fontsize=8,
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.plot(n_anh, t_eager_tn, "s", color=MAU[0], ms=7)
ax.annotate(f"thí nghiệm {n_anh:,} ảnh", (n_anh, t_eager_tn), xytext=(10, -14), textcoords="offset points", fontsize=8)
ax.set_xlabel("số ảnh đã chạy (thang log)")
ax.set_ylabel("tổng thời gian (s, thang log)")
ax.set_title("ResNet-50 theo bảng giả định của chương: ai thắng ở đâu")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Đường tức thời không có độ cao ban đầu nhưng dốc nhất, nên nó thắng ở bên trái, vùng nghiên cứu, nơi
số lần biên dịch lại lớn hơn nhiều số lần chạy sản xuất. Đường biên dịch tức thời thắng ở vùng giữa,
các lần huấn luyện dài. Đường biên dịch sâu dốc ít nhất nhưng đầu tư ban đầu lớn nhất, nên chỉ thắng sau
điểm giao thứ hai, vùng phục vụ với hàng triệu yêu cầu. Chính hai điểm giao, chứ không phải bản thân
các đường, trả lời câu hỏi nên chọn chiến lược nào. Đó cũng là căng thẳng thứ nhất khi chọn framework
mà chương nêu: nhóm nghiên cứu thử nhiều biến thể mỗi ngày không kham được vài phút biên dịch giữa hai
thí nghiệm, còn nhóm sản xuất không kham được khoản hiệu năng mất đi vì điều phối kiểu tức thời.

## Thử thêm

1. Ở phần 4, rút chuỗi xuống còn hai phép (`y = x + 0.25` rồi `torch.relu`), như Napkin Math 1.1, và
   sửa `K_PHEP = 2`. Tăng tốc đo được ở $N$ lớn bây giờ gần giá trị nào?
2. Ở phần 1, thêm `h = h * 0.5` ngay sau dòng GELU rồi chạy lại phần 1 và 2. Nhóm hợp nhất thứ nhất dài
   thêm, còn số lần nạp sau hợp nhất có đổi không? (Các `assert` về số đỉnh và số lần nạp sẽ dừng lại:
   hãy sửa chúng theo con số mới.)
3. Ở ô `breakeven`, thử GPT-2 với thời gian biên dịch 90 s và $N_{dev} = 10$. Cần bao nhiêu mẫu để lợi
   ích biên dịch vượt 1?

## Tóm lại

* Bắt đồ thị cho framework thấy cả chuỗi phép trước khi chạy, nên nó loại được mã chết và gom được các
  phép theo phần tử thành một kernel.
* Với $k$ phép theo phần tử, hợp nhất đưa lưu lượng lý tưởng từ $2kNb$ xuống $2Nb$ và số lần nạp từ $k$
  xuống 1. Tăng tốc đo được thường thấp hơn tỉ số lý tưởng đó.
* Lần theo vết chỉ ghi nhánh mà đầu vào mẫu đã đi, và vết vẫn chạy với đầu vào lẽ ra phải đi nhánh kia.
* Biên dịch có giá: lần gọi đầu, và mỗi lần một chốt bảo vệ hỏng. Nó hoàn vốn sau khoảng 134,000 ảnh
  trong kịch bản ResNet-50 của chương, nhưng làm một thí nghiệm dựng mẫu chậm đi khoảng 44 lần.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Framework — tầng giữa toán học và phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch07-frameworks/), dựng từ chương
[*ML Frameworks*](https://mlsysbook.ai/vol1/frameworks/frameworks.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_07_ml_frameworks.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.